# 【notebook｜N-16】AlphaProof V1 闭环 tiny
> 对应：《07-MCTS+V1/01-MCTS原理与V1闭环.md》【文档｜DOC-MCTS】；代码 `F-b07-run_demo`、`F-b07-self_play`、`F-b07-policy_value_eval`

- 理论：`07-MCTS+V1/01-MCTS原理与V1闭环.md` §7–§8
- 源码：`app/v1_run.py`、`v1-result/reproduction/code/src/cpu_runtime/online_ttt.py`、`nanoproof/nanoproof/search.py`
- 代码：`07-MCTS+V1/code/v1_tiny/*`
- Colab：CPU 即可（仅 torch）。

## 理论回顾

闭环 = 搜索产数据 -> 在线更新 -> 更新后策略服务后续搜索。tiny 版：MCTS 访问计数当策略目标、剩余步数当价值目标，训练 tiny 策略/价值网络。

In [1]:
# 【N-16.c1】
import math, random, torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); rng = random.Random(0)
K = 6
def problem():
    hyp = frozenset(i for i in range(K) if rng.random() < 0.7)
    goal = frozenset(i for i in range(K) if rng.random() < 0.5) or frozenset([rng.randrange(K)])
    return hyp, goal
def remaining_steps(hyp, rem): return K if any(i not in hyp for i in rem) else len(rem)
def valid_actions(hyp, rem): return [i for i in rem if i in hyp]
class Net(nn.Module):
    def __init__(self, h=128):
        super().__init__(); self.body=nn.Sequential(nn.Linear(2*K,h),nn.ReLU(),nn.Linear(h,h),nn.ReLU())
        self.pi=nn.Linear(h,K); self.v=nn.Linear(h,1)
    def enc(self, hyp, rem):
        x=torch.zeros(2*K); x[list(hyp)]=1; x[[K+a for a in rem]]=1; return x
    def forward(self,x):
        z=self.body(x); return self.pi(z), self.v(z).squeeze(-1)
net=Net(); opt=torch.optim.Adam(net.parameters(), lr=2e-3); print('device', device)

device cuda


In [2]:
# 【N-16.c2】
class Node:
    def __init__(s, state, hyp, parent=None, action=None, prior=0.0):
        s.state=state; s.hyp=hyp; s.parent=parent; s.action=action; s.prior=prior
        s.children={}; s.N=0; s.W=0.0; s.solved=(len(state)==0)
    def value(s): return 0.0 if s.N==0 else s.W/s.N
def net_priors(hyp, rem, acts):
    with torch.no_grad():
        logits, v = net(net.enc(hyp, rem).unsqueeze(0))
    p=F.softmax(logits[0, acts], -1)
    return {a: float(pp) for a, pp in zip(acts, p)}, float(v.item())
def q(ch): return 1e9 if ch.solved else -ch.value()
def mcts(hyp, goal, sims=32, c=0.2):
    root=Node(goal, hyp)
    for _ in range(sims):
        node, path = root, [root]
        while node.children:
            node = max(node.children.values(), key=lambda ch: q(ch)+c*ch.prior*math.sqrt(node.N)/(1+ch.N))
            path.append(node)
        rem=node.state; acts=valid_actions(node.hyp, rem)
        if not rem: val=0.0; node.solved=True
        elif not acts: val=float(K)
        else:
            pri, depth = net_priors(node.hyp, rem, acts)
            for a in acts:
                new=frozenset(rem-{a}); ch=Node(new, node.hyp, node, a, pri[a]); ch.solved=(len(new)==0)
                node.children[a]=ch
            val=max(depth,0.0)
        for nd in reversed(path):
            nd.N+=1; nd.W+=val
            if any(x.solved for x in nd.children.values()): nd.solved=True
        if root.solved: break
    return root

In [3]:
# 【N-16.c3】
def collect(root, out):
    stack=[root]
    while stack:
        nd=stack.pop()
        if nd.children:
            tot=sum(ch.N for ch in nd.children.values())
            if tot>0:
                out.append((nd.hyp, nd.state, {a: ch.N/tot for a,ch in nd.children.items()}, float(remaining_steps(nd.hyp,nd.state))))
            stack += list(nd.children.values())
def train(samples):
    X=torch.stack([net.enc(h,r) for h,r,_,_ in samples]); logits,v=net(X); logp=F.log_softmax(logits,-1)
    terms=[]
    for i,(h,r,d,_) in enumerate(samples):
        acts=list(d); tgt=torch.tensor([d[a] for a in acts]); terms.append(-(tgt*logp[i,acts]).sum())
    vt=torch.tensor([s[3] for s in samples], dtype=torch.float32)
    loss=torch.stack(terms).mean()+F.mse_loss(v,vt); opt.zero_grad(); loss.backward(); opt.step(); return float(loss)
for it in range(1,7):
    samples=[]; solved=0
    for _ in range(24):
        hyp,goal=problem(); root=mcts(hyp,goal); collect(root,samples); solved+=int(root.solved)
    loss=train(samples)
    print(f'iter {it} | train solved {solved}/24 | loss {loss:.3f}')
ok=0
for _ in range(40):
    hyp,goal=problem(); ok+=int(mcts(hyp,goal,64).solved)
print('final solve rate =', ok/40)

iter 1 | train solved 7/24 | loss 31.118
iter 2 | train solved 9/24 | loss 32.568
iter 3 | train solved 11/24 | loss 29.646
iter 4 | train solved 12/24 | loss 25.059
iter 5 | train solved 7/24 | loss 28.428
iter 6 | train solved 10/24 | loss 23.234
final solve rate = 0.55


/tmp/ipykernel_17168/89237986.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /app/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  loss=torch.stack(terms).mean()+F.mse_loss(v,vt); opt.zero_grad(); loss.backward(); opt.step(); return float(loss)


## 思考题
1. 为什么这里只需要 OR（顺序关闭原子），而真实 Lean 证明需要 AND + focus？
2. 访问计数当策略目标、剩余步数当价值目标，对应 `online_ttt.py` 的哪些字段？
3. `post_update_generations=0` 说明闭环哪里断了？
4. 把探索系数 `c` 调大/调小，solve rate 如何变化？

## 预期输出
每轮 `train solved/24` 与 `final solve rate` 上升；`loss` 下降。